# Lesson 1 - Tabular classification with an MLP neural network

> This notebook is an educational demonstration and does not constitute a medical diagnostic system.

## What is a neural network?

An **artificial neural network** is a computational model inspired by how the human brain works. It consists of basic units called **artificial neurons** (or nodes), organized into **layers** that process information sequentially.

An MLP (Multilayer Perceptron) is one of the most fundamental neural network architectures. It consists of:

- an **input layer**, which receives the raw data (features);
- one or more **hidden layers**, which learn intermediate representations of the data;
- an **output layer**, which produces the final prediction.

<mark>An MLP is an ideal starting point for understanding neural networks because all the essential concepts—neurons, weights, biases, activations, and backpropagation—are present in a direct and transparent way.</mark>

### The artificial neuron

Each neuron performs a simple operation in two stages:

**1. Linear combination** — multiplies each input by its weight and adds a bias:

$$z = w_1 x_1 + w_2 x_2 + \ldots + w_n x_n + b = \mathbf{w} \cdot \mathbf{x} + b$$

**2. Activation function** — applies a nonlinear transformation to the result:

$$\hat{y} = f(z)$$

The **weights** and **biases** are the parameters learned by the network during training. The **activation function** allows the network to learn nonlinear relationships in the data; without it, stacking layers would provide no benefit because the result would always remain a linear function.

### Activation functions

The most common activation functions in modern neural networks include:

| Function | Formula | Typical use |
|----------|---------|-------------|
| **ReLU** | $f(z) = \max(0, z)$ | Hidden layers (current default) |
| **Sigmoid** | $f(z) = \frac{1}{1 + e^{-z}}$ | Binary output (probability) |
| **Softmax** | $f(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$ | Multiclass output |
| **Tanh** | $f(z) = \tanh(z)$ | Recurrent layers |

This notebook uses **ReLU** in the hidden layers. It is simple, efficient, and helps mitigate the vanishing-gradient problem that affects Sigmoid in deep networks.

### How the network learns — Backpropagation

Training a neural network involves a repeated cycle called an **epoch**:

1. **Forward pass** — data flows through the network from input to output, producing a prediction $\hat{y}$.
2. **Error calculation** — the **loss function** measures how wrong the prediction was: $\mathcal{L}(\hat{y}, y)$.
3. **Backward pass** — **backpropagation** calculates the gradient of the loss with respect to each parameter using the chain rule from calculus.
4. **Weight update** — the **optimizer** (for example, Adam or SGD) adjusts the weights in the direction opposite to the gradient to reduce the error.

$$w \leftarrow w - \eta \cdot \frac{\partial \mathcal{L}}{\partial w}$$

Here, $\eta$ is the **learning rate**, a hyperparameter that controls the size of each update step.

PyTorch automates steps 3 and 4 with `loss.backward()` and `optimizer.step()`.

### What does the partial derivative in the loss function mean?

**What is a partial derivative?**<br />
A derivative measures: *“if I change this variable slightly, how much does the function change?”*

When a function depends on **multiple variables**, the **partial derivative** with respect to one variable measures the effect of changing only that variable while keeping all others fixed. The symbol $\partial$ (read “partial”) indicates this, in contrast to the $d$ used for an ordinary derivative.

**Partial with respect to what?**<br />
The loss function $\mathcal{L}$ depends simultaneously on **all network weights**, potentially thousands or millions of parameters: $w_1, w_2, \ldots, w_n$.

$$\frac{\partial \mathcal{L}}{\partial w}$$

This means: *“while keeping all other weights fixed, if I change this weight $w$ slightly, how much does the loss increase or decrease?”*

The answer is a number: the **local gradient** for that weight. If it is positive, increasing $w$ increases the error; if it is negative, increasing $w$ decreases the error.

**Why subtract?**<br />
The **minus sign** is the key: moving $w$ in the direction opposite to the gradient makes the loss decrease. This is **gradient descent**—moving downhill on the error surface.

$\eta$ (the learning rate) controls the step size. If it is too large, the model may jump over the minimum; if it is too small, convergence may be slow.

### Further reading

For a visual and intuitive explanation of how each component works, see:

- [But what is a neural network? — 3Blue1Brown](https://www.youtube.com/watch?v=aircAruvnKk)
- [Gradient descent, how neural networks learn — 3Blue1Brown](https://www.youtube.com/watch?v=IHZwWFHWa-w)
- [Backpropagation calculus — 3Blue1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8)

## Classification with an MLP

### Dataset

We use the same [Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) dataset prepared in the previous notebook.

Here, however, we use the **numeric features** extracted from the images—radius, texture, perimeter, area, and others—instead of the synthetic report text. This makes the problem simpler and more direct, which is ideal for demonstrating an MLP without the complexity of natural language processing.

### Data preparation

Data preparation is performed in the [Data preparation](01-prepare-data.en.ipynb) notebook.

The preparation notebook generates three Parquet files. This classifier uses the first one, `data/breast_cancer.parquet`, which contains only the original tabular data, without synthetic reports or noise. The file must exist at the project root before this notebook is run.

### Check available software and hardware

In [ ]:
from pathlib import Path
import random
import numpy as np
import torch

def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Project root not found.")

PROJECT_ROOT = find_project_root()

# Fixing random sources makes weight initialization and training reproducible
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("PyTorch version: ", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
print("CUDA version: ", torch.version.cuda)

### Load the data

In [ ]:
import pandas as pd

# Numeric features used as network inputs
FEATURE_COLUMNS = [
    "radius", "texture", "perimeter", "area",
    "smoothness", "compactness", "concavity",
    "concave points", "symmetry", "fractal dimension"
]

def preprocess_dataset(path):
    df = pd.read_parquet(path)

    # Encode the diagnosis: Malignant=1, Benign=0
    df['diagnosis_encoded'] = df['diagnosis'].apply(lambda x: 1 if x == 'M' else 0)

    return df

# First Parquet file generated by the preparation notebook
DATASET_PATH = PROJECT_ROOT / "data" / "breast_cancer.parquet"
data = preprocess_dataset(DATASET_PATH)

data[['id_number', 'diagnosis', 'diagnosis_encoded'] + FEATURE_COLUMNS].head()

### Diagnosis distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Count samples in each class
data["diagnosis"].value_counts()

In [ ]:
sns.pairplot(data, hue='diagnosis', vars=['radius', 'texture', 'perimeter', 'area'], palette='Set2')
plt.suptitle("Attribute Distribution by Diagnosis", y=1.02)
plt.show()

### Data normalization

The features have very different scales (for example, `area` can be 1,000 times larger than `smoothness`). Neural networks are sensitive to input scale, and gradients from features with very large values could dominate learning.

**Standardization** addresses this issue: `StandardScaler` subtracts the mean and divides by the standard deviation, giving each feature a mean of 0 and a standard deviation of 1.

> **Important:** the scaler must be fitted **only on the training data** and then applied, without refitting, to the validation and test data. This prevents data leakage.

### Split the data into training, validation, and test sets

The **training set** is used to adjust the network weights; the **validation set** monitors training and guides model choices; and the **test set** remains untouched until the final evaluation. The split is stratified to approximately preserve the proportion of benign and malignant cases in all three sets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = data[FEATURE_COLUMNS].values
y = data['diagnosis_encoded'].values

# Reserve 15% for testing; this set is used only during final evaluation
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)

# Reserve another 15% of the total for validation (0.15 / 0.85 of the remaining set)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15 / 0.85,
    random_state=SEED, stratify=y_train_val
)

# Standardize: fit only on training data and apply the same transformation elsewhere
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"Training:  {X_train.shape[0]} samples")
print(f"Validation:{X_val.shape[0]} samples")
print(f"Test:      {X_test.shape[0]} samples")
print(f"Features:  {X_train.shape[1]}")

### Create datasets and DataLoaders

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Convert NumPy arrays to PyTorch tensors
train_inputs  = torch.tensor(X_train, dtype=torch.float32)
train_labels  = torch.tensor(y_train, dtype=torch.long)
val_inputs    = torch.tensor(X_val,   dtype=torch.float32)
val_labels    = torch.tensor(y_val,   dtype=torch.long)
test_inputs   = torch.tensor(X_test,  dtype=torch.float32)
test_labels   = torch.tensor(y_test,  dtype=torch.long)

# Group features and labels into datasets
train_dataset = TensorDataset(train_inputs, train_labels)
val_dataset   = TensorDataset(val_inputs,   val_labels)
test_dataset  = TensorDataset(test_inputs,  test_labels)

batch_size = 16

# DataLoaders iterate over the data in batches during training
generator = torch.Generator().manual_seed(SEED)
train_dataloader = DataLoader(
    train_dataset, shuffle=True, batch_size=batch_size, generator=generator
)
val_dataloader  = DataLoader(val_dataset,  shuffle=False, batch_size=batch_size)
test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)

### Model definition

The architecture is intentionally small for educational purposes:

```text
Input (10 features)
    ↓
Linear layer:  10 → 16  +  ReLU
    ↓
Linear layer:  16 →  8  +  ReLU
    ↓
Linear layer:   8 →  2  (logits: Benign / Malignant)
```

**10 (input)**<br />
The 10 numeric features extracted from the images, including radius, texture, perimeter, and area.

**10 → 16 (expansion)**<br />
The network needs additional “space” to learn combinations of the 10 original features. With only 10 neurons in the hidden layer, each neuron could become roughly responsible for one feature, leaving less capacity to learn interactions such as “large radius and irregular texture.” Expanding to 16 provides room for richer intermediate representations.

**16 → 8 (compression)**<br />
After expansion, compression forces the network to distill what it has learned. It must discard noise and preserve information that discriminates between classes. This intentional bottleneck resembles the encoder concept in autoencoders. Empirically, this hourglass pattern can generalize better than keeping every layer the same size.

**8 → 2 (output)**<br />
One raw score, or logit, for each class.

**The general pattern is:**<br />
input → expansion → compression → output

**About the layer sizes**<br />
The specific values 16 and 8 are heuristic. Powers of two are conventional and work well with hardware, while reducing each layer by roughly half is a common rule of thumb. There is no exact formula; in production, these values would be hyperparameters selected through cross-validation.

Equally valid alternatives include 10→32→16→2 for greater capacity or 10→8→2 for a simpler model, depending on problem complexity. For 569 samples and 10 features, the current architecture is already expressive without being unnecessarily large.

In PyTorch, models are defined as classes that inherit from `nn.Module`. The `forward` method describes how data flows through the network during the forward pass.

In [ ]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 16),  # Layer 1: input_size → 16 neurons
            nn.ReLU(),                  # ReLU activation
            nn.Linear(16, 8),           # Layer 2: 16 → 8 neurons
            nn.ReLU(),                  # ReLU activation
            nn.Linear(8, 2)             # Output layer: 8 → 2 classes
        )

    def forward(self, x):
        return self.network(x)


input_size = len(FEATURE_COLUMNS)  # 10 features
model = MLP(input_size)

print(model)
print(f"\nTotal trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

### Logits, loss function, and optimizer

The final layer produces two **logits**, which are raw scores—one for each class. They are not probabilities and do not need to sum to 1.

`CrossEntropyLoss` receives these logits directly and internally applies `LogSoftmax` and `NLLLoss`. Therefore, we **must not add `Softmax` to the final layer or apply it before the loss**. `argmax` can select a class directly from the logits; `Softmax` is used only after training when probabilities need to be displayed.

In [ ]:
# CrossEntropyLoss combines Softmax and NLLLoss and is suitable for multiclass classification
criterion = nn.CrossEntropyLoss()

# Adam is an adaptive optimizer that often converges faster than plain SGD
# lr (learning rate): size of each weight update step
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### Model training

In [ ]:
from tqdm import tqdm

# Select the device: GPU when available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

epochs = 10

# History for later visualization
history = {"loss": [], "val_accuracy": []}

for epoch in tqdm(range(epochs), desc="Training"):

    # =================== Training ===================
    model.train()  # Enable training mode (dropout, batch normalization, and so on)
    total_loss = 0

    for batch_inputs, batch_labels in train_dataloader:

        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)

        # 1. Clear gradients accumulated during the previous step
        optimizer.zero_grad()

        # 2. Forward pass: calculate predictions
        logits = model(batch_inputs)

        # 3. Calculate the loss
        loss = criterion(logits, batch_labels)

        # 4. Backward pass: calculate gradients through backpropagation
        loss.backward()

        # 5. Update weights with the optimizer
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataloader)

    # =================== Validation ===================
    model.eval()  # Disable training-only behavior
    total_correct = 0
    total_samples = 0

    with torch.no_grad():  # Disable gradient calculation for faster evaluation
        for batch_inputs, batch_labels in val_dataloader:

            batch_inputs = batch_inputs.to(device)
            batch_labels = batch_labels.to(device)

            logits      = model(batch_inputs)
            predictions = torch.argmax(logits, dim=-1)

            total_correct += (predictions == batch_labels).sum().item()
            total_samples += batch_labels.size(0)

    val_accuracy = total_correct / total_samples

    history["loss"].append(avg_loss)
    history["val_accuracy"].append(val_accuracy)

    tqdm.write(f"Epoch {epoch + 1:02d}/{epochs}  |  Loss: {avg_loss:.4f}  |  Val Accuracy: {val_accuracy:.2%}")

### Learning curves

The learning curves show how the network evolves across epochs:

- **Training loss** should gradually decrease. A smooth decrease indicates that the optimizer is moving toward a minimum in a stable way.
- **Validation accuracy** may appear stable even while the loss decreases because accuracy is discrete (each error is worth about 0.88% in this dataset). A decreasing loss can mean that the model is becoming **more confident** in its correct predictions, not only making more correct predictions.

In [ ]:
epoch_range = range(1, epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# =================== Loss =================== 
ax1.plot(epoch_range, history["loss"], color="steelblue", linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss by epoch (training)")
ax1.grid(True, alpha=0.3)

# =================== Accuracy ===================
ax2.plot(epoch_range, [v * 100 for v in history["val_accuracy"]], color="seagreen", linewidth=2)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy by epoch (validation)")
ax2.set_ylim(90, 101)
ax2.grid(True, alpha=0.3)

plt.suptitle("Learning Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Final evaluation on the test set

Only now is the test set presented to the model. It did not participate in weight fitting, normalization, or epoch monitoring.

In [ ]:
test_targets       = []
test_predictions   = []
test_probabilities = []

model.eval()

with torch.no_grad():
    for batch_inputs, batch_labels in test_dataloader:

        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)

        logits = model(batch_inputs)

        test_targets.extend(batch_labels.cpu().numpy())

        preds = torch.argmax(logits, dim=-1)
        test_predictions.extend(preds.cpu().numpy())

        # Probability of the positive class (Malignant = 1)
        test_probabilities.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())

#### Accuracy, precision, recall, and F1

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy  = accuracy_score(test_targets, test_predictions)
precision = precision_score(test_targets, test_predictions)
recall    = recall_score(test_targets, test_predictions)
f1        = f1_score(test_targets, test_predictions)
roc_auc   = roc_auc_score(test_targets, test_probabilities)

print(f"Accuracy:   {accuracy:.2f}")
print(f"Precision:  {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1 Score:  {f1:.2f}")
print(f"ROC AUC:   {roc_auc:.2f}")

#### Classification report

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    test_targets, test_predictions, target_names=["Benign", "Malignant"]
))

#### Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix

sns.heatmap(
    confusion_matrix(test_targets, test_predictions),
    annot=True, fmt='d', cmap='Blues',
    xticklabels=["Benign", "Malignant"],
    yticklabels=["Benign", "Malignant"]
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

#### Precision–recall curve

In [ ]:
from sklearn.metrics import precision_recall_curve

precision_curve, recall_curve, _ = precision_recall_curve(
    test_targets, test_probabilities
)

plt.plot(recall_curve, precision_curve)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision–Recall Curve')
plt.show()

#### ROC curve

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(test_targets, test_probabilities)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0, 1], [0, 1], 'k--', label="Random")
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

### Comparison with a linear baseline

Logistic regression is a simple linear classifier that serves as a **baseline**: a minimum reference for evaluating whether the neural network's additional complexity provides a benefit. Both models use exactly the same normalized features and test set.

If their results are similar, it does not mean the MLP failed. It means this dataset may be separated effectively by an approximately linear decision boundary. This is an important reminder that neural networks are not automatically the best solution for every problem.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(max_iter=1000, random_state=SEED)
logistic_model.fit(X_train, y_train)

logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "Accuracy": [
        accuracy,
        accuracy_score(y_test, logistic_predictions),
    ],
    "Precision": [
        precision,
        precision_score(y_test, logistic_predictions),
    ],
    "Recall": [
        recall,
        recall_score(y_test, logistic_predictions),
    ],
    "F1": [
        f1,
        f1_score(y_test, logistic_predictions),
    ],
    "ROC AUC": [
        roc_auc,
        roc_auc_score(y_test, logistic_probabilities),
    ],
}, index=["MLP", "Logistic Regression"])

comparison.round(3)

### Save the trained model

The checkpoint contains the MLP weights and the metadata required to apply exactly the same preparation during future inference.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "lesson-01"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = OUTPUT_DIR / "simple-mlp-classifier.pt"

checkpoint = {
    "model_state_dict": model.state_dict(),
    "input_size": input_size,
    "feature_columns": FEATURE_COLUMNS,
    "class_names": ["Benign", "Malignant"],
    "scaler_mean": scaler.mean_,
    "scaler_scale": scaler.scale_,
    "seed": SEED,
}

torch.save(checkpoint, checkpoint_path)
print(f"Checkpoint saved to: {checkpoint_path}")

### Usage example

In [ ]:
import numpy as np

# New data: [radius, texture, perimeter, area, smoothness,
#               compactness, concavity, concave_points, symmetry, fractal_dimension]
new_samples = np.array([
    # Typical benign profile: small radius and smooth texture
    [12.0, 15.0, 78.0, 450.0, 0.09, 0.07, 0.03, 0.02, 0.18, 0.06],
    # Typical malignant profile: large radius and irregular texture
    [22.0, 28.0, 145.0, 1500.0, 0.15, 0.25, 0.30, 0.15, 0.28, 0.09],
])

# Apply the same standardization used during training
new_samples_scaled = scaler.transform(new_samples)
new_tensor = torch.tensor(new_samples_scaled, dtype=torch.float32).to(device)

# Prediction
model.eval()
with torch.no_grad():
    logits      = model(new_tensor)
    probs       = torch.softmax(logits, dim=1)
    predictions = torch.argmax(logits, dim=-1)

label_map = {0: "Benign", 1: "Malignant"}

for i, (pred, prob) in enumerate(zip(predictions, probs)):
    print(f"Sample {i + 1}: {label_map[pred.item()]}  "
          f"(Benign: {prob[0]:.1%}  |  Malignant: {prob[1]:.1%})")